# Version 0.6


# LLM Benchmark on Google Colab T4 GPU - MMLU

This notebook demonstrates how to:
1. Load a pre-trained LLM model
2. Run inference benchmarks on Google Colab's T4 GPU
3. Measure performance metrics (latency, throughput, memory usage)
4. Visualize the results

The model used is **Mistral 7B Instruct v0.3** loaded in 4-bit so it can run on a Colab T4 GPU.
The benchmark dataset is **MMLU** (Massive Multitask Language Understanding) across all available subjects, with a configurable per-subject sample cap to keep runtime practical.


## 1. Install Required Libraries


In [ ]:
# Install required libraries
import subprocess
import sys

# Install transformers and torch
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

print("✓ All libraries installed successfully!")


## 2. Check GPU and Setup


In [ ]:
import os
import torch
import numpy as np
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Check GPU availability
print("GPU Information:")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Current GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")


## 3. Load Pretrained LLM Model

We use **Mistral 7B Instruct v0.3** in a quantized configuration. That keeps the notebook usable on a T4 while switching from classifier-style inference to prompt-based multiple-choice scoring.


In [ ]:
# Load Mistral 7B Instruct v0.3 tokenizer and model
print("Loading Mistral 7B Instruct v0.3 model...")
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_quantization = torch.cuda.is_available()
if use_quantization:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        token=hf_token,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(model_name, token=hf_token)
    model = model.to(device)

model.eval()  # Set to evaluation mode

total_parameters = sum(p.numel() for p in model.parameters())
param_size = total_parameters * (0.5 if use_quantization else 4) / (1024 ** 2)

print(f"✓ Model loaded successfully!")
print(f"  Model: {model_name}")
print(f"  Total parameters: {total_parameters / 1e9:.2f}B")
print(f"  Approx. model size: {param_size:.2f} MB")


## 4. Load MMLU (Massive Multitask Language Understanding) Dataset

The MMLU dataset contains multiple-choice questions across diverse domains from elementary mathematics to professional medicine. It includes 57 subjects spanning STEM, social sciences, and humanities.

To keep the benchmark practical on a Colab T4 GPU, this notebook evaluates a configurable number of questions per subject.


In [ ]:
from datasets import load_dataset

# Get all MMLU subjects
mmlu_subjects = [
    "abstract_algebra", "anatomy", "astronomy", "business_ethics",
    "clinical_knowledge", "college_biology", "college_chemistry",
    "college_computer_science", "college_mathematics", "college_medicine",
    "college_physics", "computer_security", "conceptual_physics",
    "econometrics", "electrical_engineering", "elementary_mathematics",
    "formal_logic", "global_facts", "high_school_biology",
    "high_school_chemistry", "high_school_computer_science",
    "high_school_european_history", "high_school_geography",
    "high_school_government_and_politics", "high_school_macroeconomics",
    "high_school_mathematics", "high_school_microeconomics",
    "high_school_physics", "high_school_psychology", "high_school_statistics",
    "high_school_us_history", "high_school_world_history", "human_aging",
    "human_sexuality", "international_law", "jurisprudence",
    "logical_fallacies", "machine_learning", "management", "marketing",
    "medical_genetics", "medical_knowledge", "miscellaneous",
    "moral_disputes", "moral_scenarios", "nutrition", "philosophy",
    "prehistory", "professional_accounting", "professional_law",
    "professional_medicine", "professional_psychology", "public_relations",
    "security_studies", "sociology", "us_foreign_policy", "virology",
    "world_religions"
]

MAX_QUESTIONS_PER_SUBJECT = 10

print(f"Loading MMLU dataset with {len(mmlu_subjects)} subjects...\n")
print(f"Using up to {MAX_QUESTIONS_PER_SUBJECT} questions per subject for a practical Colab run.\n")

mmlu_data = []
failed_subjects = []

for subject in mmlu_subjects:
    try:
        dataset = load_dataset("cais/mmlu", subject, split="test", trust_remote_code=True)
        sample_count = min(MAX_QUESTIONS_PER_SUBJECT, len(dataset))
        dataset = dataset.select(range(sample_count))

        for sample in dataset:
            mmlu_data.append({
                'subject': subject,
                'question': sample['question'],
                'choices': sample['choices'],
                'answer': chr(ord('A') + sample['answer'])
            })
        print(f"  ✓ {subject}: {len(dataset)} questions")
    except Exception as e:
        print(f"  ✗ {subject}: Failed to load")
        failed_subjects.append(subject)

print(f"\n✓ Loaded {len(mmlu_data)} total MMLU questions from {len(mmlu_subjects) - len(failed_subjects)} subjects")

if mmlu_data:
    print(f"\nFirst example:")
    sample = mmlu_data[0]
    print(f"  Subject: {sample['subject']}")
    print(f"  Question: {sample['question']}")
    print(f"  Choices: {sample['choices']}")
    print(f"  Answer: {sample['answer']}")


## 5. Prepare Question-Choice Pairs


In [ ]:
# Flatten questions into question-choice samples for scoring
mmlu_samples = []

for question_index, sample in enumerate(mmlu_data):
    question_text = sample['question']
    choices = sample['choices']
    correct_answer = sample['answer']
    subject = sample['subject']

    option_lines = "\n".join(
        f"{chr(ord('A') + choice_idx)}. {choice_text}"
        for choice_idx, choice_text in enumerate(choices)
    )
    prompt = (
        "You are solving a multiple-choice question.\n"
        f"Question: {question_text}\n"
        f"Options:\n{option_lines}\n\n"
        "Reply with only the answer letter."
    )

    for choice_idx, choice_text in enumerate(choices):
        choice_letter = chr(ord('A') + choice_idx)  # A, B, C, D

        mmlu_samples.append({
            'question_id': f"mmlu_{question_index}",
            'subject': subject,
            'question': question_text,
            'choices': choices,
            'choice_idx': choice_idx,
            'choice_letter': choice_letter,
            'choice_text': choice_text,
            'completion': f" {choice_letter}",
            'prompt': prompt,
            'label': 1 if choice_letter == correct_answer else 0,
            'answer_key': correct_answer
        })

print(f"Created {len(mmlu_samples)} question-choice pairs from {len(mmlu_data)} questions")
print(f"Unique subjects: {len(set(s['subject'] for s in mmlu_samples))}")


## 6. Run MMLU Benchmark

Evaluate the model on MMLU questions - measure accuracy across all subjects.


In [ ]:
# Evaluate model on MMLU dataset
from sklearn.metrics import accuracy_score
from collections import defaultdict

print(f"Running MMLU Benchmark...\n")
print("=" * 60)

# Store predictions per question and per subject
question_predictions = defaultdict(list)
subject_accuracies = defaultdict(lambda: {'correct': 0, 'total': 0})
timings = []

print(f"Processing {len(mmlu_samples)} question-choice pairs...")
print(f"Batch processing with batch size: 8\n")

# Process in batches
batch_size = 8

for batch_idx in range(0, len(mmlu_samples), batch_size):
    batch_samples = mmlu_samples[batch_idx:batch_idx + batch_size]
    batch_texts = [sample['prompt'] + sample['completion'] for sample in batch_samples]
    prompt_lengths = [len(tokenizer(sample['prompt']).input_ids) for sample in batch_samples]
    full_lengths = [len(tokenizer(text).input_ids) for text in batch_texts]

    # Tokenize
    encodings = tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    # Measure inference time
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)
        log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)
        target_ids = input_ids[:, 1:]
        token_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_time = time.time()
    timings.append(end_time - start_time)

    # Store scores grouped by question
    for i, sample in enumerate(batch_samples):
        prompt_length = prompt_lengths[i]
        full_length = full_lengths[i]
        completion_length = max(full_length - prompt_length, 1)
        start_index = max(prompt_length - 1, 0)
        end_index = min(start_index + completion_length, token_log_probs.shape[1])
        score = token_log_probs[i, start_index:end_index].sum().item()

        question_predictions[sample['question_id']].append({
            'choice_idx': sample['choice_idx'],
            'choice_letter': sample['choice_letter'],
            'confidence': score,
            'label': sample['label'],
            'subject': sample['subject'],
            'answer_key': sample['answer_key']
        })

print(f"\n✓ Inference complete!")
print(f"Total inference time: {sum(timings):.2f}s")
print(f"Average time per sample: {(sum(timings) / len(mmlu_samples) * 1000):.2f}ms")

# Determine best answer per question and calculate accuracy
predictions_per_question = []
labels_per_question = []
all_subjects = []

for q_id, choices in question_predictions.items():
    # Find choice with highest confidence
    best_choice = max(choices, key=lambda x: x['confidence'])
    predicted_choice = best_choice['choice_letter']
    ground_truth = best_choice['answer_key']
    subject = best_choice['subject']

    predictions_per_question.append(predicted_choice)
    labels_per_question.append(ground_truth)
    all_subjects.append(subject)

    # Update subject statistics
    subject_accuracies[subject]['total'] += 1
    if predicted_choice == ground_truth:
        subject_accuracies[subject]['correct'] += 1

# Calculate metrics
accuracy_overall = accuracy_score(labels_per_question, predictions_per_question)

print(f"\nOverall Accuracy Metrics:")
print(f"  Overall Accuracy: {accuracy_overall * 100:.2f}%")

# Convert defaultdict to regular dict for storage
subject_accuracies = dict(subject_accuracies)

benchmark_results = {
    'Overall Accuracy': accuracy_overall,
    'Total Questions': len(predictions_per_question),
    'Total Samples': len(mmlu_samples),
    'Avg Inference Time (ms)': (sum(timings) / len(mmlu_samples)) * 1000,
    'Total Inference Time (s)': sum(timings),
    'Subject Accuracies': subject_accuracies,
    'Questions per Subject': MAX_QUESTIONS_PER_SUBJECT,
}

print("\n✓ Benchmark complete!")


## 7. Display Results

Show detailed benchmark results including accuracy scores and performance metrics.


In [ ]:
# Display detailed results
print("\n" + "=" * 100)
print("MMLU BENCHMARK RESULTS - Mistral 7B Instruct v0.3 (All Subjects)")
print("=" * 100)

print(f"\nDataset Statistics:")
print(f"  Total Questions: {benchmark_results['Total Questions']}")
print(f"  Total Samples: {benchmark_results['Total Samples']}")
print(f"  Total Subjects: {len(subject_accuracies)}")
print(f"  Questions per Subject: {benchmark_results['Questions per Subject']}")

print(f"\nOverall Accuracy Score:")
print(f"  Overall Accuracy: {benchmark_results['Overall Accuracy'] * 100:.2f}%")

print(f"\nAccuracy by Subject:")
print("-" * 100)
print(f"{'Subject':<40} {'Accuracy':<15} {'Correct/Total':<20}")
print("-" * 100)

for subject, stats in sorted(benchmark_results['Subject Accuracies'].items(),
                              key=lambda x: (x[1]['correct'] / x[1]['total']) if x[1]['total'] > 0 else 0,
                              reverse=True):
    if stats['total'] > 0:
        subject_acc = (stats['correct'] / stats['total']) * 100
        print(f"{subject:<40} {subject_acc:>6.2f}%{'':<7} {stats['correct']:>2}/{stats['total']:<10}")

print("-" * 100)

print(f"\nPerformance Metrics:")
print(f"  Avg Inference Time: {benchmark_results['Avg Inference Time (ms)']:.2f} ms")
print(f"  Total Inference Time: {benchmark_results['Total Inference Time (s)']:.2f} s")
print(f"  Samples per Second: {benchmark_results['Total Samples'] / benchmark_results['Total Inference Time (s)']:.2f}")

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1024 ** 2
    current_memory = torch.cuda.memory_allocated() / 1024 ** 2
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024 ** 2

    print(f"\nGPU Memory Usage:")
    print(f"  Peak Memory: {peak_memory:.2f} MB")
    print(f"  Current Memory: {current_memory:.2f} MB")
    print(f"  Total GPU Memory: {total_memory:.2f} MB")
    print(f"  Memory Utilization: {(peak_memory / total_memory) * 100:.2f}%")

print(f"\nModel Information:")
print(f"  Model: {model_name}")
print(f"  Total Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"  Approx. Model Size: {param_size:.2f} MB")

print("\n" + "=" * 100)


## 8. Visualize Results

Create plots to display benchmark results for easy interpretation.


In [ ]:
# Create detailed visualizations for MMLU benchmark results with individual subject plots
# Prepare subject data
subjects_list = []
accuracies_list = []
correct_counts = []
total_counts = []

for subject, stats in sorted(benchmark_results['Subject Accuracies'].items()):
    if stats['total'] > 0:
        subjects_list.append(subject)
        acc = (stats['correct'] / stats['total']) * 100
        accuracies_list.append(acc)
        correct_counts.append(stats['correct'])
        total_counts.append(stats['total'])

num_subjects = len(subjects_list)
num_cols = 3
num_rows = (num_subjects + num_cols - 1) // num_cols + 2  # +2 for overall plots

fig = plt.figure(figsize=(20, 4 * num_rows))
gs = fig.add_gridspec(num_rows, num_cols, hspace=0.4, wspace=0.3)

fig.suptitle("Mistral 7B Instruct v0.3 Performance on MMLU - Detailed Subject Analysis (Sampled Subjects)",
             fontsize=20, fontweight="bold", y=0.995)

# Plot 1: Overall Accuracy (top-left)
ax_overall = fig.add_subplot(gs[0, 0])
overall_acc = benchmark_results['Overall Accuracy'] * 100
colors_overall = ['steelblue']
bars_overall = ax_overall.bar(['Overall Accuracy'], [overall_acc],
                               color=colors_overall, alpha=0.7, edgecolor='black', linewidth=1.5)
ax_overall.set_ylabel('Accuracy (%)', fontsize=12, fontweight="bold")
ax_overall.set_title('Overall MMLU Accuracy', fontsize=13, fontweight="bold")
ax_overall.set_ylim([0, 100])
ax_overall.grid(True, alpha=0.3, axis='y')
for bar, acc in zip(bars_overall, [overall_acc]):
    height = bar.get_height()
    ax_overall.text(bar.get_x() + bar.get_width()/2., height, f'{acc:.1f}%',
                   ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 2: Subject Accuracies Overview (top-middle and top-right)
ax_overview = fig.add_subplot(gs[0, 1:])
if subjects_list:
    colors_subj = plt.cm.RdYlGn(np.array(accuracies_list) / 100.0)
    bars_overview = ax_overview.barh(range(len(subjects_list)), accuracies_list,
                                      color=colors_subj, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax_overview.set_xlabel('Accuracy (%)', fontsize=12, fontweight="bold")
    ax_overview.set_title('Accuracy Rankings by Subject', fontsize=13, fontweight="bold")
    ax_overview.set_yticks(range(len(subjects_list)))
    ax_overview.set_yticklabels(subjects_list, fontsize=9)
    ax_overview.set_xlim([0, 100])
    ax_overview.grid(True, alpha=0.3, axis='x')

    # Add value labels
    for i, (bar, acc, correct, total) in enumerate(zip(bars_overview, accuracies_list, correct_counts, total_counts)):
        width = bar.get_width()
        ax_overview.text(width, bar.get_y() + bar.get_height()/2.,
                        f' {acc:.1f}% ({correct}/{total})',
                        ha='left', va='center', fontsize=8, fontweight='bold')
else:
    ax_overview.text(0.5, 0.5, 'No subject data available',
                    ha='center', va='center', transform=ax_overview.transAxes, fontsize=12)
    ax_overview.axis('off')

# Plot individual subject plots
for idx, (subject, accuracy, correct, total) in enumerate(zip(subjects_list, accuracies_list, correct_counts, total_counts)):
    row = 1 + (idx // num_cols)
    col = idx % num_cols
    ax = fig.add_subplot(gs[row, col])

    # Pie chart for correct/incorrect
    sizes = [correct, total - correct]
    labels = [f'Correct {correct}', f'Incorrect {total - correct}']
    colors_pie = ['#2ecc71', '#e74c3c']
    explode = (0.05, 0.05)
    wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                                        autopct='%1.1f%%', shadow=True, startangle=90,
                                        textprops={'fontsize': 9, 'weight': 'bold'})

    # Make percentage text more visible
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(10)
        autotext.set_weight('bold')

    # Title with subject and stats
    ax.set_title(f'{subject} | {accuracy:.1f}% Accuracy (n={total})',
                fontsize=10, fontweight="bold", pad=10)

plt.show()

print()
print("✓ Detailed Visualization complete!")


## About This Benchmark

### What is MMLU?
The **Massive Multitask Language Understanding** (MMLU) benchmark is one of the most comprehensive evaluation suites for language models. It includes 57 tasks covering:
- **STEM**: Mathematics, Physics, Chemistry, Biology, Computer Science
- **Social Sciences**: History, Geography, Government, Economics, Law
- **Humanities**: Philosophy, Religion, Languages
- **Other**: Business, Medicine, Psychology, Nutrition, Virology, and more

### Dataset Details
- **Total Questions**: 14,042 multiple-choice questions
- **Number of Subjects**: 57
- **Format**: Multiple choice with 4 answer options (A, B, C, D)
- **Coverage**: Professional and academic knowledge across diverse fields
- **This Benchmark**: Uses a configurable number of test questions per subject to keep runtime practical on Colab T4

### How This Benchmark Works
1. Each question is rendered as a multiple-choice prompt
2. Mistral scores the answer-letter completions for the four options
3. The highest-scoring answer letter becomes the prediction
4. Accuracy is computed per subject and overall across the sampled MMLU questions
5. The results are visualized for easy comparison across subjects
